# Agent: Camera Detector (Phase 6, MQTT Subscribe)

This notebook implements the camera detector agent:
- subscribes to `simcity/surveillance/events/illegal`
- applies strict step-window filtering (`message.step == local_current_step`)
- maps event coordinates into a 100x100 camera grid with clamping
- publishes camera decisions to `simcity/surveillance/detections/camera`
- publishes one-shot alarms (`ttl_steps=1`) to `simcity/surveillance/alarms/active`


## Live test flow
1. Run setup cells in this detector notebook first.
2. Run the producer notebook simulation cell.
3. Return here and run the status cell to inspect processed/dropped counts.
4. Run cleanup when done.


In [ ]:
from __future__ import annotations
import json
import math
import random
import threading
import time
from collections import deque
from typing import Any
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher


In [ ]:
cfg = load_config()
sim_cfg = cfg.simulation
SURVEILLANCE_TOPIC_ROOT = "simcity/surveillance"
TOPIC_EVENTS_ILLEGAL = f"{SURVEILLANCE_TOPIC_ROOT}/events/illegal"
TOPIC_DETECTIONS = f"{SURVEILLANCE_TOPIC_ROOT}/detections/camera"
TOPIC_ALARMS = f"{SURVEILLANCE_TOPIC_ROOT}/alarms/active"
LIVE_QOS = 0
LIVE_RETAIN = False
STEP_DELAY_S = float(sim_cfg.step_delay_s) if (sim_cfg is not None and sim_cfg.step_delay_s > 0) else 1.0
ALARM_TTL_STEPS = 1
print(f"Detector MQTT broker: {cfg.mqtt.host}:{cfg.mqtt.port}")
print(f"Topics -> sub: {TOPIC_EVENTS_ILLEGAL}, pub: {TOPIC_DETECTIONS}, {TOPIC_ALARMS}")


In [ ]:
GRID_SIZE = 100
CELL_SIZE_M = 10.0
TOTAL_CELLS = GRID_SIZE * GRID_SIZE
COVERED_CELLS_TARGET = int(TOTAL_CELLS * 0.35)
def clamp_cell_index(value: int) -> int:
    return max(0, min(GRID_SIZE - 1, value))
def xy_to_cell(x: float, y: float) -> tuple[int, int]:
    cell_x = clamp_cell_index(math.floor(float(x) / CELL_SIZE_M))
    cell_y = clamp_cell_index(math.floor(float(y) / CELL_SIZE_M))
    return cell_x, cell_y
all_cells = [(cell_x, cell_y) for cell_x in range(GRID_SIZE) for cell_y in range(GRID_SIZE)]
camera_covered_cells = set(random.sample(all_cells, COVERED_CELLS_TARGET))
print(f"Camera coverage initialized: {len(camera_covered_cells)}/{TOTAL_CELLS} cells")


In [ ]:
mqtt_connector: MqttConnector | None = None
mqtt_publisher: MqttPublisher | None = None
subscription_enabled = False
anchor_step: int | None = None
anchor_started_at: float | None = None
processed_event_ids: set[str] = set()
stats = {"received": 0, "processed": 0, "detected_true": 0, "duplicates": 0, "dropped_late": 0, "dropped_out_of_order": 0, "dropped_invalid": 0}
recent_decisions: deque[dict[str, Any]] = deque(maxlen=20)
state_lock = threading.Lock()
def current_local_step_unlocked() -> int | None:
    if anchor_step is None or anchor_started_at is None:
        return None
    elapsed = time.perf_counter() - anchor_started_at
    return anchor_step + int(elapsed // STEP_DELAY_S)
def process_illegal_event(payload: dict[str, Any]) -> None:
    global anchor_step, anchor_started_at
    required = {"step", "event_id", "human_id", "x", "y", "event_type"}
    if not required.issubset(payload.keys()) or payload.get("event_type") != "illegal_act":
        with state_lock:
            stats["dropped_invalid"] += 1
        return
    try:
        msg_step = int(payload["step"])
        event_id = str(payload["event_id"])
        human_id = str(payload["human_id"])
        x = float(payload["x"])
        y = float(payload["y"])
    except (TypeError, ValueError):
        with state_lock:
            stats["dropped_invalid"] += 1
        return
    with state_lock:
        if anchor_step is None:
            anchor_step = msg_step
            anchor_started_at = time.perf_counter()
        local_current_step = current_local_step_unlocked()
        if local_current_step is None:
            stats["dropped_invalid"] += 1
            return
        if msg_step < local_current_step:
            stats["dropped_late"] += 1
            return
        if msg_step > local_current_step:
            stats["dropped_out_of_order"] += 1
            return
        if event_id in processed_event_ids:
            stats["duplicates"] += 1
            return
        processed_event_ids.add(event_id)
        camera_cell = xy_to_cell(x, y)
        detected = camera_cell in camera_covered_cells
        detection_payload = {"step": msg_step, "event_id": event_id, "human_id": human_id, "camera_cell": [camera_cell[0], camera_cell[1]], "detected": detected}
        if mqtt_publisher is not None:
            mqtt_publisher.publish_json(TOPIC_DETECTIONS, json.dumps(detection_payload), qos=LIVE_QOS, retain=LIVE_RETAIN)
            if detected:
                alarm_payload = {"step": msg_step, "event_id": event_id, "human_id": human_id, "x": x, "y": y, "ttl_steps": ALARM_TTL_STEPS}
                mqtt_publisher.publish_json(TOPIC_ALARMS, json.dumps(alarm_payload), qos=LIVE_QOS, retain=LIVE_RETAIN)
        stats["processed"] += 1
        if detected:
            stats["detected_true"] += 1
        recent_decisions.append({"step": msg_step, "event_id": event_id, "camera_cell": [camera_cell[0], camera_cell[1]], "detected": detected, "ttl_steps": ALARM_TTL_STEPS if detected else None})
def on_illegal_event_message(client, userdata, message):
    try:
        payload = json.loads(message.payload.decode("utf-8"))
    except Exception:
        with state_lock:
            stats["dropped_invalid"] += 1
        return
    with state_lock:
        stats["received"] += 1
    process_illegal_event(payload)
try:
    mqtt_connector = MqttConnector(cfg.mqtt, client_id_suffix="camera-detector")
    mqtt_connector.connect()
    if mqtt_connector.wait_for_connection(timeout=5.0):
        mqtt_publisher = MqttPublisher(mqtt_connector)
        mqtt_connector.client.on_message = on_illegal_event_message
        mqtt_connector.client.subscribe(TOPIC_EVENTS_ILLEGAL, qos=LIVE_QOS)
        subscription_enabled = True
        print(f"Subscribed to {TOPIC_EVENTS_ILLEGAL}. Waiting for producer events...")
    else:
        print("MQTT connection timeout. Detector stays idle until connection is available.")
except Exception as exc:
    print(f"MQTT unavailable ({exc}). Detector remains idle safely.")


In [ ]:
def print_detector_status() -> None:
    with state_lock:
        local_step = current_local_step_unlocked()
        snapshot = dict(stats)
        recent = list(recent_decisions)
    print("Detector status:")
    print(f"- subscription_enabled={subscription_enabled}")
    print(f"- local_current_step={local_step}")
    print(f"- received={snapshot['received']}")
    print(f"- processed={snapshot['processed']}")
    print(f"- detected_true={snapshot['detected_true']}")
    print(f"- dropped_late={snapshot['dropped_late']}")
    print(f"- dropped_out_of_order={snapshot['dropped_out_of_order']}")
    print(f"- duplicates={snapshot['duplicates']}")
    print(f"- dropped_invalid={snapshot['dropped_invalid']}")
    if recent:
        print("Recent decisions (latest up to 5):")
        for decision in recent[-5:]:
            print(f"  - {decision}")
print_detector_status()


In [ ]:
boundary_samples = [(0.0, 0.0), (9.99, 9.99), (10.0, 10.0), (999.9, 999.9), (1000.0, 1000.0), (-0.1, 500.0), (500.0, 1000.5)]
print("Boundary mapping samples:")
for x, y in boundary_samples:
    print(f"- (x={x}, y={y}) -> cell={xy_to_cell(x, y)}")


In [ ]:
if mqtt_connector is not None and mqtt_connector.client.is_connected():
    mqtt_connector.disconnect()
    print("Camera detector MQTT connector disconnected.")
else:
    print("Camera detector cleanup: no active MQTT connection.")
